In [ ]:
pip install -U spacy

In [ ]:
pip install spacy_transformers

## Preparing data by merging the json files and also garding only the the skill annotalions

In [10]:
import os
import json

def load_spacy_data(folder_path):
    training_data = []

    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        text = data["text"]
        annotations = data["annotations"]

        entities = []

        for ann in annotations:
            start, end, label = ann

            # KEEP ONLY SKILLS
            if "SKILL" in label:
                # clean label if needed (optional)
                entities.append((start, end, "SKILL"))

        # skip empty samples (important!)
        if entities:
            training_data.append((text, {"entities": entities}))

    return training_data

In [11]:
folder_path = r"C:\Users\wiame\Desktop\career-platform\ml\cv_parser\ResumesJsonAnnotated"
data = load_spacy_data(folder_path)

print(data[0])

('One97 Communications Limited \nData Scientist Jan 2019 to Till Date \nDetect important information from images and redact\nrequired fields. YOLO CNN Object-detection, OCR\nInsights, find anomaly or performance drop in all\npossible sub-space. \nPredict the Insurance claim probability. Estimate the\npremium amount to be charged\nB.Tech(Computer Science) from SGBAU university in\n2017. \nM.Tech (Computer Science Engineering) from Indian\nInstitute of Technology (IIT), Kanpur in 2019WORK EXPERIENCE\nEDUCATIONMACY WILLIAMS\nDATA SCIENTIST\nData Scientist working  on problems related to market research and customer analysis. I want to expand my arsenal of\napplication building and work on different kinds of problems. Looking for a role where I can work with a coordinative team\nand exchange knowledge during the process.\nJava, C++, Python, Machine Learning, Algorithms, Natural Language Processing, Deep Learning, Computer Vision, Pattern\nRecognition, Data Science, Data Analysis, Software 

In [12]:
import spacy 
from spacy.tokens import DocBin
from tqdm import tqdm
import json

In [13]:
spacy.__version__

'3.8.14'

In [14]:
!nvidia-smi

Sat Apr 25 02:36:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.95                 Driver Version: 581.95         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 500 Ada Gener...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P8              4W /   25W |       1MiB /   4094MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
len(data)

4971

In [16]:
!python -m spacy init fill-config config/base_config.cfg config/config.cfg

[!] Nothing to auto-fill: base config is already complete
[+] Saved config
config\config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


## preparing the data to be in a spacy format 

In [18]:
pip install ftfy

Note: you may need to restart the kernel to use updated packages.


In [45]:
import re

def clean_text(text):
    """Only removes invalid characters — never shifts offsets."""
    if not isinstance(text, str):
        text = str(text)
    # Strip surrogates (crash fix) and null bytes only
    text = text.encode('utf-8', errors='ignore').decode('utf-8', errors='ignore')
    text = text.replace('\x00', '')  # null bytes
    return text


def get_spacy_doc(file, data):
    nlp = spacy.blank("en")
    db = DocBin()

    for text, annot in tqdm(data):
        text = clean_text(text)          # safe: no offset shifting
        doc = nlp.make_doc(text)
        entities = annot["entities"]
        ents = []
        entity_indices = set()

        for start, end, label in entities:
            # ── Trim whitespace at character level (before char_span) ────────
            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            # ── Skip empty spans ─────────────────────────────────────────────
            if start >= end:
                file.write(f"[EMPTY] label={label} | {repr(text[max(0,start-20):end+20])}\n")
                continue

            # ── Skip overlapping spans ───────────────────────────────────────
            span_indices = set(range(start, end))
            if span_indices & entity_indices:
                file.write(f"[OVERLAP] ({start},{end}) label={label}\n")
                continue
            entity_indices.update(span_indices)

            span = doc.char_span(start, end, label=label, alignment_mode="contract")

            if span is None:
                file.write(f"[NULL] ({start},{end}) label={label} | {repr(text[max(0,start-20):end+20])}\n")
                continue

            # ── Final whitespace guard (token level) ─────────────────────────
            if span.text != span.text.strip():
                file.write(f"[WHITESPACE TOKEN] {repr(span.text)} label={label}\n")
                continue

            ents.append(span)

        try:
            doc.ents = ents
            db.add(doc)
        except Exception as e:
            file.write(f"[DOC ERROR] {e} | {repr(text[:80])}\n")

    return db

In [46]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [47]:
from sklearn.model_selection import train_test_split 

train,test = train_test_split(data,test_size=0.3)

In [48]:
len(train),len(test)

(3479, 1492)

### saving the train and test data 

### training the model 

In [49]:
from pathlib import Path

BASE = Path(r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model')

with open(BASE / 'train_file.txt', 'w', encoding='utf-8') as file:

    print("Processing training data...")
    train_db = get_spacy_doc(file, train)
    train_db.to_disk(BASE / 'train_data.spacy')
    print(f"✔ Saved train_data.spacy ({len(train)} docs)")

    print("Processing test data...")
    test_db = get_spacy_doc(file, test)
    test_db.to_disk(BASE / 'test_data.spacy')
    print(f"✔ Saved test_data.spacy ({len(test)} docs)")

print("✔ Done. Check train_file.txt for any skipped spans.")

Processing training data...


100%|██████████| 3479/3479 [00:41<00:00, 84.78it/s] 


✔ Saved train_data.spacy (3479 docs)
Processing test data...


100%|██████████| 1492/1492 [00:19<00:00, 76.05it/s] 


✔ Saved test_data.spacy (1492 docs)
✔ Done. Check train_file.txt for any skipped spans.


In [ ]:
python -m spacy train "C:/Users/wiame/Desktop/career-platform/ml/cv_parser/config/config.cfg" --output "C:/Users/wiame/Desktop/career-platform/ml/cv_parser/model/output" --paths.train "C:/Users/wiame/Desktop/career-platform/ml/cv_parser/model/train_data.spacy" --paths.dev "C:/Users/wiame/Desktop/career-platform/ml/cv_parser/model/test_data.spacy" 

In [68]:
nlp = spacy.load(r'C:\Users\wiame\Desktop\career-platform\ml\cv_parser\model\output\model-best')

In [64]:
pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.


In [69]:
import sys,fitz

fname = r"C:\Users\wiame\Desktop\career-platform\ml\cv_parser\test\john_smith_cv.pdf"
doc = fitz.open(fname)

## Extract text content from pdf 

In [70]:
text = " "
for page in doc:
    text = text + str(page.get_text())
text 

" John Smith\nSenior Software Engineer\njohn.smith@email.com | +1 (555) 012-3456 | linkedin.com/in/johnsmith | github.com/johnsmith\nSan Francisco, CA, United States\nPROFESSIONAL SUMMARY\nSoftware Engineer with 7 years of experience designing and delivering scalable web applications, machine\nlearning pipelines, and cloud-native infrastructure. Proven track record of leading cross-functional teams,\noptimizing system performance, and shipping high-impact products at scale. Strong communicator with deep\nexpertise in Python, distributed systems, and DevOps practices.\nTECHNICAL SKILLS\nLanguages:\nPython, JavaScript, TypeScript, Java, SQL, Bash\nFrameworks:\nDjango, FastAPI, React, Node.js, TensorFlow, PyTorch, Scikit-learn\nDatabases:\nPostgreSQL, MongoDB, Redis, MySQL, Elasticsearch\nCloud & DevOps:\nAWS (EC2, S3, Lambda, RDS), Docker, Kubernetes, Terraform, CI/CD, Linux\nTools & Methods:\nGit, GitHub Actions, Jenkins, REST APIs, Microservices, Agile, Scrum\nML & Data:\nNLP, Computer

## Testing the model 

In [71]:
doc = nlp(text)
for ent in doc.ents:
    print(ent.text,">>>>>>>>>",ent.label_)

Senior >>>>>>>>> SKILL
Software Engineer >>>>>>>>> SKILL
San >>>>>>>>> SKILL
PROFESSIONAL >>>>>>>>> SKILL
Software Engineer >>>>>>>>> SKILL
7 years of experience >>>>>>>>> SKILL
designing >>>>>>>>> SKILL
delivering >>>>>>>>> SKILL
scalable >>>>>>>>> SKILL
web >>>>>>>>> SKILL
applications >>>>>>>>> SKILL
learning >>>>>>>>> SKILL
pipelines >>>>>>>>> SKILL
cloud >>>>>>>>> SKILL
infrastructure >>>>>>>>> SKILL
Proven track record >>>>>>>>> SKILL
leading >>>>>>>>> SKILL
functional >>>>>>>>> SKILL
teams >>>>>>>>> SKILL
optimizing >>>>>>>>> SKILL
performance >>>>>>>>> SKILL
shipping >>>>>>>>> SKILL
impact >>>>>>>>> SKILL
products >>>>>>>>> SKILL
scale >>>>>>>>> SKILL
expertise >>>>>>>>> SKILL
Python >>>>>>>>> SKILL
distributed >>>>>>>>> SKILL
DevOps >>>>>>>>> SKILL
practices >>>>>>>>> SKILL
TECHNICAL SKILLS >>>>>>>>> SKILL
Languages >>>>>>>>> SKILL
Python >>>>>>>>> SKILL
JavaScript >>>>>>>>> SKILL
Java >>>>>>>>> SKILL
SQL >>>>>>>>> SKILL
Bash >>>>>>>>> SKILL
Frameworks >>>>>>>>> SKILL
Django >